In [4]:
import os
import cv2
import time
import shutil
import numpy as np

from ultralytics import YOLO
from pathlib import Path


In [5]:
# PATHS
RAW_DATASET = Path("dataset/train")
PREPROCESSED_DATASET = Path("dataset/train_preprocessed23431434113")

PREPROCESSED_DATASET.mkdir(parents=True, exist_ok=True)

### Предобработка кадров  

Приведение к пиксельным кадрам

In [7]:
import shutil
import numpy as np

# =========================
# PREPROCESSING FUNCTIONS
# =========================

def adaptive_gamma(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = gray.mean()

    if mean < 80:      # ночь
        gamma = 1.6
    elif mean < 120:   # сумерки
        gamma = 1.3
    else:              # день
        gamma = 0.9

    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)


def clahe_luminance(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def silhouette_boost(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    grad_x = cv2.Sobel(gray, cv2.CV_16S, 1, 0)
    grad_y = cv2.Sobel(gray, cv2.CV_16S, 0, 1)

    abs_x = cv2.convertScaleAbs(grad_x)
    abs_y = cv2.convertScaleAbs(grad_y)

    edges = cv2.addWeighted(abs_x, 0.5, abs_y, 0.5, 0)
    edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX)

    mask = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    return cv2.addWeighted(img, 0.9, mask, 0.1, 0)


def mild_sharpen(img):
    kernel = np.array([[0, -0.5, 0],
                       [-0.5, 3, -0.5],
                       [0, -0.5, 0]])
    return cv2.filter2D(img, -1, kernel)

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def gamma_correction(img, gamma):
    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)


def add_noise(img, sigma=12):
    noise = np.random.normal(0, sigma, img.shape).astype(np.uint8)
    return cv2.add(img, noise)


def edge_enhance(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    return cv2.addWeighted(img, 0.8, edges, 0.2, 0)


def sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


def full_preprocess(img):
    img = apply_clahe(img)
    img = gamma_correction(img, gamma=np.random.uniform(1.2, 1.6))
    img = add_noise(img)
    img = edge_enhance(img)
    img = sharpen(img)
    return img


# =========================
# DATASET PREPROCESSING
# =========================
print("🔧 Предобработка датасета...")

for img_file in RAW_DATASET.glob("*.jpg"):
    txt_file = img_file.with_suffix(".txt")
    if not txt_file.exists():
        continue

    img = cv2.imread(str(img_file))
    if img is None:
        continue

    processed = full_preprocess(img)

    cv2.imwrite(str(PREPROCESSED_DATASET / img_file.name), processed)
    shutil.copy(txt_file, PREPROCESSED_DATASET / txt_file.name)

print("Предобработка завершена")

🔧 Предобработка датасета...
Предобработка завершена
